# V3-3 — Resumable A2-MP hard-negative mining

This notebook scores only sliding windows from train videos whose video-level label is zero. It never reads validation windows for mining and does not train a new model.

In [2]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import time

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
import torch
from torch import nn
from torchvision import models
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\NexarCollisionData')
MANIFEST_ROOT = DATA_ROOT / 'manifests_v3'
PROCESSED_ROOT = DATA_ROOT / 'processed_v3'
PREDICTION_ROOT = DATA_ROOT / 'predictions_v3'
REPORT_ROOT = DATA_ROOT / 'reports_v3'
MODEL_ROOT = DATA_ROOT / 'models_v2'
for directory in [PROCESSED_ROOT, PREDICTION_ROOT, REPORT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

SEQUENCE_MANIFEST_PATH = MANIFEST_ROOT / 'sequence_manifest_v3_sliding.csv'
A2MP_CHECKPOINT_PATH = MODEL_ROOT / 'resnet18_meanmax_pooling_frozen_multipos_best.pt'
REGISTRY_PATH = REPORT_ROOT / 'experiments_v3_registry.csv'

PARTIAL_FEATURES_PATH = PROCESSED_ROOT / 'a2mp_hn_train_negative_features_partial.pt'
FEATURES_PATH = PROCESSED_ROOT / 'a2mp_hn_train_negative_features.pt'
SCORES_PATH = PREDICTION_ROOT / 'a2mp_train_negative_window_scores_v3.csv'
FAILURES_PATH = REPORT_ROOT / 'a2mp_hard_negative_decode_failures_v3.csv'
HARD_NEGATIVES_PATH = MANIFEST_ROOT / 'hard_negatives_round1.csv'
REVIEW_QUEUE_PATH = REPORT_ROOT / 'hard_negative_review_queue_v3.csv'
CONTACT_SHEET_PATH = REPORT_ROOT / 'hard_negative_contact_sheet_v3.jpg'
SUMMARY_PATH = REPORT_ROOT / 'hard_negative_mining_summary_v3.json'

WINDOW_SECONDS = 5.0
NUM_FRAMES = 16
TARGET_HEIGHT = 224
TARGET_WIDTH = 320
FEATURE_DIM = 512
SEQUENCE_BATCH_SIZE = 2
FRAME_BATCH_SIZE = 32
SAVE_EVERY_BATCHES = 25
HARD_NEGATIVE_THRESHOLD = 0.60
FALLBACK_TOP_K_PER_VIDEO = 3
MIN_SELECTED_HARD_NEGATIVES = 240
MAX_SELECTED_PER_VIDEO = 3
CONTACT_SHEET_LIMIT = 50
RUN_FULL_MINING = True
MAX_SEQUENCES = None

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
PREPROCESSING_VERSION = 'v2_multipos_rgb_letterbox_replicate_224x320'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for item in [SEQUENCE_MANIFEST_PATH, A2MP_CHECKPOINT_PATH, REGISTRY_PATH]:
    assert item.is_file(), 'Missing prerequisite: {}'.format(item)
print({'device': str(device), 'run_full_mining': RUN_FULL_MINING, 'sequence_batch_size': SEQUENCE_BATCH_SIZE, 'frame_batch_size': FRAME_BATCH_SIZE})


{'device': 'cpu', 'run_full_mining': True, 'sequence_batch_size': 2, 'frame_batch_size': 32}


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1) Preflight and strict train-negative eligibility checks
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

sequence_manifest = pd.read_csv(SEQUENCE_MANIFEST_PATH).copy()
sequence_manifest['video_id'] = sequence_manifest['video_id'].astype(str)
negative_windows = sequence_manifest.loc[
    sequence_manifest['split'].eq('train')
    & sequence_manifest['video_label'].eq(0)
    & sequence_manifest['window_role'].eq('negative_video')
].copy()
negative_windows = negative_windows.sort_values(['video_id', 'window_index'], key=lambda values: values.astype(int) if values.name == 'video_id' else values).reset_index(drop=True)
assert len(negative_windows) > 0
assert negative_windows['sequence_id'].is_unique
assert negative_windows['split'].eq('train').all()
assert negative_windows['video_label'].eq(0).all()
assert negative_windows['window_role'].eq('negative_video').all()
assert negative_windows['time_of_event'].isna().all()
assert negative_windows['time_of_alert'].isna().all()
assert negative_windows['video_path'].map(lambda value: Path(value).is_file()).all()
assert negative_windows.groupby('video_id')['split'].nunique().eq(1).all()

MANIFEST_SHA256 = sha256_file(SEQUENCE_MANIFEST_PATH)
CHECKPOINT_SHA256 = sha256_file(A2MP_CHECKPOINT_PATH)
print({'negative_train_windows': len(negative_windows), 'negative_train_videos': negative_windows['video_id'].nunique(), 'manifest_sha256': MANIFEST_SHA256, 'checkpoint_sha256': CHECKPOINT_SHA256})


{'negative_train_windows': 3497, 'negative_train_videos': 240, 'manifest_sha256': '44df8aade328d3993ba9568f401337a2462370e4ab4a63b8ffd2f88ebe5830f8', 'checkpoint_sha256': 'f4ae77cbacb466bcdd6ea4f11951f1f585d4ab6060c30d6bff3e135dce9020bb'}


In [4]:
# 2) Exact frozen A2-MP model and preprocessing contract
class ResNet18MeanMaxPoolingHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(nn.LayerNorm(FEATURE_DIM * 2), nn.Dropout(0.35), nn.Linear(FEATURE_DIM * 2, 1))
    def forward(self, sequence_features):
        mean_features = sequence_features.mean(dim=1)
        max_features = sequence_features.max(dim=1).values
        return self.classifier(torch.cat([mean_features, max_features], dim=1)).squeeze(1)

weights = models.ResNet18_Weights.IMAGENET1K_V1
backbone = models.resnet18(weights=weights)
encoder = nn.Sequential(*list(backbone.children())[:-1]).to(device).eval()
for parameter in encoder.parameters():
    parameter.requires_grad_(False)
head = ResNet18MeanMaxPoolingHead().to(device).eval()
checkpoint = torch.load(A2MP_CHECKPOINT_PATH, map_location=device, weights_only=False)
head.load_state_dict(checkpoint['model_state_dict'])
CHECKPOINT_SIGNATURE = 'epoch={}|pr_auc={:.12f}|sha256={}'.format(checkpoint['epoch'], checkpoint['validation_pr_auc'], CHECKPOINT_SHA256)

def resize_letterbox_rgb(image_rgb):
    height, width = image_rgb.shape[:2]
    scale = min(TARGET_WIDTH / width, TARGET_HEIGHT / height)
    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(image_rgb, (new_width, new_height), interpolation=interpolation)
    pad_x, pad_y = TARGET_WIDTH - new_width, TARGET_HEIGHT - new_height
    left, right = pad_x // 2, pad_x - pad_x // 2
    top, bottom = pad_y // 2, pad_y - pad_y // 2
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_REPLICATE)

def normalized_tensor(image_rgb):
    tensor = torch.from_numpy(image_rgb.copy()).permute(2, 0, 1).float().div_(255.0)
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD

def decode_rgb_at_timestamp(cap, timestamp, fps):
    step = 1.0 / fps if np.isfinite(fps) and fps > 0 else 1.0 / 30.0
    for attempt, offset in enumerate((0.0, step, -step, 2 * step)):
        target = max(0.0, float(timestamp) + offset)
        cap.set(cv2.CAP_PROP_POS_MSEC, target * 1000.0)
        ok, frame_bgr = cap.read()
        if ok and frame_bgr is not None:
            return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB), 'exact' if attempt == 0 else 'seek_fallback_{}'.format(attempt)
    return None, 'decode_failed'

def decode_sequence(row):
    cap = cv2.VideoCapture(str(row.video_path))
    if not cap.isOpened():
        return None, {'sequence_id': row.sequence_id, 'video_id': row.video_id, 'video_path': row.video_path, 'window_start': row.window_start, 'window_end': row.window_end, 'error_reason': 'cannot_open'}
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    timestamps = np.linspace(float(row.window_start), float(row.window_end), num=NUM_FRAMES, endpoint=False)
    frames, statuses, previous_frame = [], [], None
    try:
        for timestamp in timestamps:
            frame_rgb, status = decode_rgb_at_timestamp(cap, timestamp, fps)
            if frame_rgb is None and previous_frame is not None:
                frame_rgb, status = previous_frame.copy(), 'repeated_previous_after_decode_failure'
            if frame_rgb is None:
                return None, {'sequence_id': row.sequence_id, 'video_id': row.video_id, 'video_path': row.video_path, 'window_start': row.window_start, 'window_end': row.window_end, 'error_reason': status}
            previous_frame = frame_rgb
            frames.append(normalized_tensor(resize_letterbox_rgb(frame_rgb)))
            statuses.append(status)
    finally:
        cap.release()
    return torch.stack(frames), {'sequence_id': row.sequence_id, 'decode_status': ';'.join(sorted(set(statuses))), 'valid_frames': NUM_FRAMES}

with torch.inference_mode():
    preflight_start = time.perf_counter()
    preflight_tensors, preflight_details = [], []
    for _, row in negative_windows.head(2).iterrows():
        tensor, details = decode_sequence(row)
        assert tensor is not None, details
        preflight_tensors.append(tensor)
        preflight_details.append(details)
    preflight_images = torch.cat(preflight_tensors, dim=0).to(device)
    preflight_features = encoder(preflight_images).flatten(1).reshape(2, NUM_FRAMES, FEATURE_DIM)
    preflight_probabilities = torch.sigmoid(head(preflight_features)).cpu().numpy()
    preflight_seconds = time.perf_counter() - preflight_start
assert preflight_features.shape == (2, NUM_FRAMES, FEATURE_DIM)
assert np.isfinite(preflight_probabilities).all()
print({'checkpoint_signature': CHECKPOINT_SIGNATURE, 'preflight_seconds_for_2_windows': round(preflight_seconds, 3), 'estimated_minutes_for_all_windows': round(preflight_seconds * len(negative_windows) / 2 / 60, 1), 'preflight_probabilities': preflight_probabilities.tolist()})


{'checkpoint_signature': 'epoch=12|pr_auc=0.730450347449|sha256=f4ae77cbacb466bcdd6ea4f11951f1f585d4ab6060c30d6bff3e135dce9020bb', 'preflight_seconds_for_2_windows': 13.627, 'estimated_minutes_for_all_windows': 397.1, 'preflight_probabilities': [0.1597863882780075, 0.15387769043445587]}


In [ ]:
# 3) Resumable frozen feature extraction for every negative train window
def load_partial_features():
    if not PARTIAL_FEATURES_PATH.is_file():
        return {}
    payload = torch.load(PARTIAL_FEATURES_PATH, map_location='cpu', weights_only=False)
    if payload.get('manifest_sha256') != MANIFEST_SHA256 or payload.get('checkpoint_sha256') != CHECKPOINT_SHA256 or payload.get('preprocessing_version') != PREPROCESSING_VERSION:
        return {}
    return {sequence_id: feature.float().cpu() for sequence_id, feature in payload.get('features_by_sequence', {}).items() if tuple(feature.shape) == (NUM_FRAMES, FEATURE_DIM)}

def save_partial_features(features_by_sequence):
    temporary_path = PARTIAL_FEATURES_PATH.with_suffix('.tmp')
    torch.save({'manifest_sha256': MANIFEST_SHA256, 'checkpoint_sha256': CHECKPOINT_SHA256, 'preprocessing_version': PREPROCESSING_VERSION, 'features_by_sequence': features_by_sequence}, temporary_path)
    temporary_path.replace(PARTIAL_FEATURES_PATH)

features_by_sequence = load_partial_features()
expected_sequence_ids = negative_windows['sequence_id'].tolist()
features_by_sequence = {sequence_id: feature for sequence_id, feature in features_by_sequence.items() if sequence_id in set(expected_sequence_ids)}
failures = []
if RUN_FULL_MINING:
    missing = negative_windows.loc[~negative_windows['sequence_id'].isin(features_by_sequence)].copy()
    if MAX_SEQUENCES is not None:
        missing = missing.head(int(MAX_SEQUENCES))
    print('Cached features: {} / {}; missing in this run: {}'.format(len(features_by_sequence), len(expected_sequence_ids), len(missing)))
    batches_since_save = 0
    with torch.inference_mode():
        for batch_start in tqdm(range(0, len(missing), SEQUENCE_BATCH_SIZE), desc='A2-MP negative-window features'):
            batch_rows = [row for _, row in missing.iloc[batch_start:batch_start + SEQUENCE_BATCH_SIZE].iterrows()]
            valid_tensors, valid_rows = [], []
            for row in batch_rows:
                tensor, details = decode_sequence(row)
                if tensor is None:
                    failures.append(details)
                else:
                    valid_tensors.append(tensor)
                    valid_rows.append(row)
            if valid_tensors:
                image_batch = torch.cat(valid_tensors, dim=0)
                feature_chunks = []
                for frame_start in range(0, len(image_batch), FRAME_BATCH_SIZE):
                    feature_chunks.append(encoder(image_batch[frame_start:frame_start + FRAME_BATCH_SIZE].to(device)).flatten(1).cpu())
                batch_features = torch.cat(feature_chunks, dim=0).reshape(len(valid_tensors), NUM_FRAMES, FEATURE_DIM)
                for row, feature in zip(valid_rows, batch_features):
                    features_by_sequence[row.sequence_id] = feature
            batches_since_save += 1
            if batches_since_save >= SAVE_EVERY_BATCHES:
                save_partial_features(features_by_sequence)
                batches_since_save = 0
    save_partial_features(features_by_sequence)

missing_after_run = [sequence_id for sequence_id in expected_sequence_ids if sequence_id not in features_by_sequence]
failure_table = pd.DataFrame(failures, columns=['sequence_id', 'video_id', 'video_path', 'window_start', 'window_end', 'error_reason'])
failure_table.to_csv(FAILURES_PATH, index=False)
print({'features_available': len(features_by_sequence), 'expected_features': len(expected_sequence_ids), 'missing_after_run': len(missing_after_run), 'failures_this_run': len(failure_table), 'failure_report': str(FAILURES_PATH)})


Cached features: 500 / 3497; missing in this run: 2997


A2-MP negative-window features:  55%|█████▌    | 829/1499 [2:56:19<2:20:33, 12.59s/it]

In [ ]:
# 4) Score windows and choose hard negatives without touching validation
mining_complete = len(features_by_sequence) == len(expected_sequence_ids)
if mining_complete:
    ordered_features = torch.stack([features_by_sequence[sequence_id] for sequence_id in expected_sequence_ids])
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(ordered_features), 256):
            probabilities.append(torch.sigmoid(head(ordered_features[start:start + 256].to(device))).cpu())
    scored_windows = negative_windows.copy()
    scored_windows['positive_probability'] = torch.cat(probabilities).numpy()
    scored_windows['checkpoint_signature'] = CHECKPOINT_SIGNATURE
    scored_windows['manifest_sha256'] = MANIFEST_SHA256
    scored_windows['preprocessing_version'] = PREPROCESSING_VERSION
    scored_windows['rank_in_video'] = scored_windows.groupby('video_id')['positive_probability'].rank(method='first', ascending=False).astype(int)
    scored_windows['rank_global'] = scored_windows['positive_probability'].rank(method='first', ascending=False).astype(int)
    scored_windows.sort_values(['positive_probability', 'video_id', 'window_index'], ascending=[False, True, True]).to_csv(SCORES_PATH, index=False)

    threshold_candidates = scored_windows.loc[scored_windows['positive_probability'].ge(HARD_NEGATIVE_THRESHOLD)].sort_values(['video_id', 'positive_probability'], ascending=[True, False])
    selected = threshold_candidates.groupby('video_id', group_keys=False).head(MAX_SELECTED_PER_VIDEO).copy()
    selected['selection_reason'] = 'probability_gte_threshold'
    if len(selected) < MIN_SELECTED_HARD_NEGATIVES:
        fallback = scored_windows.sort_values(['video_id', 'positive_probability'], ascending=[True, False]).groupby('video_id', group_keys=False).head(FALLBACK_TOP_K_PER_VIDEO).copy()
        fallback = fallback.loc[~fallback['sequence_id'].isin(selected['sequence_id'])]
        needed = MIN_SELECTED_HARD_NEGATIVES - len(selected)
        fallback = fallback.sort_values(['positive_probability', 'video_id'], ascending=[False, True]).head(needed)
        fallback['selection_reason'] = 'fallback_top_k_per_video'
        selected = pd.concat([selected, fallback], ignore_index=True)
    selected = selected.sort_values(['positive_probability', 'video_id'], ascending=[False, True]).copy()
    selected['is_hard_negative'] = True
    selected['taxonomy'] = 'unreviewed'
    selected['sample_weight_hn1'] = 1.5
    selected.to_csv(HARD_NEGATIVES_PATH, index=False)
    selected.head(CONTACT_SHEET_LIMIT).to_csv(REVIEW_QUEUE_PATH, index=False)

    complete_payload = {'manifest_sha256': MANIFEST_SHA256, 'checkpoint_sha256': CHECKPOINT_SHA256, 'preprocessing_version': PREPROCESSING_VERSION, 'sequence_ids': expected_sequence_ids, 'features': ordered_features}
    torch.save(complete_payload, FEATURES_PATH)
    print({'mining_complete': mining_complete, 'scored_windows': len(scored_windows), 'threshold_candidates': len(threshold_candidates), 'selected_hard_negatives': len(selected), 'score_path': str(SCORES_PATH), 'hard_negatives_path': str(HARD_NEGATIVES_PATH)})
else:
    scored_windows = pd.DataFrame()
    selected = pd.DataFrame()
    print('Mining remains resumable. Re-run this notebook to continue from the partial feature cache.')


In [ ]:
# 5) Contact sheet, summary and registry
def read_contact_thumbnail(row, width=256, height=144):
    cap = cv2.VideoCapture(str(row.video_path))
    if not cap.isOpened():
        return np.full((height, width, 3), 30, dtype=np.uint8)
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    timestamp = (float(row.window_start) + float(row.window_end)) / 2.0
    frame_rgb, _ = decode_rgb_at_timestamp(cap, timestamp, fps)
    cap.release()
    if frame_rgb is None:
        return np.full((height, width, 3), 30, dtype=np.uint8)
    thumbnail = cv2.resize(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR), (width, height), interpolation=cv2.INTER_AREA)
    text = 'p={:.3f} | {} | {:.1f}-{:.1f}s'.format(float(row.positive_probability), row.video_id, float(row.window_start), float(row.window_end))
    cv2.putText(thumbnail, text, (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
    return thumbnail

if mining_complete and len(selected):
    contact_rows = []
    selected_preview = selected.head(CONTACT_SHEET_LIMIT).reset_index(drop=True)
    for start in range(0, len(selected_preview), 5):
        thumbnails = [read_contact_thumbnail(row) for _, row in selected_preview.iloc[start:start + 5].iterrows()]
        while len(thumbnails) < 5:
            thumbnails.append(np.full_like(thumbnails[0], 30))
        contact_rows.append(cv2.hconcat(thumbnails))
    cv2.imwrite(str(CONTACT_SHEET_PATH), cv2.vconcat(contact_rows), [cv2.IMWRITE_JPEG_QUALITY, 95])

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'completed' if mining_complete else 'partial_resumable',
    'input_scope': 'train_negative_video_windows_only',
    'negative_train_windows_expected': int(len(expected_sequence_ids)),
    'features_available': int(len(features_by_sequence)),
    'decode_failures_this_run': int(len(failure_table)),
    'hard_negative_threshold': HARD_NEGATIVE_THRESHOLD,
    'fallback_top_k_per_video': FALLBACK_TOP_K_PER_VIDEO,
    'max_selected_per_video': MAX_SELECTED_PER_VIDEO,
    'selected_hard_negatives': int(len(selected)),
    'checkpoint_signature': CHECKPOINT_SIGNATURE,
    'manifest_sha256': MANIFEST_SHA256,
    'preprocessing_version': PREPROCESSING_VERSION,
    'preflight_seconds_for_2_windows': float(preflight_seconds),
    'contact_sheet': str(CONTACT_SHEET_PATH) if mining_complete and len(selected) else ''
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

registry = pd.read_csv(REGISTRY_PATH)
registry_row = {'run_id': 'V3_03_HARD_NEGATIVE_MINING_R1', 'stage': 'V3-3 hard_negative_mining', 'model_id': 'A2-MP frozen', 'dataset_version': 'v3_sliding_core_context_v1', 'split_version': 'train_negative_only', 'window_version': '5s_stride2.5s', 'feature_version': 'a2mp_hn_negative_features_v1', 'augmentation_version': 'none', 'checkpoint_path': str(A2MP_CHECKPOINT_PATH), 'config_path': 'notebooks/33_v3_hard_negative_mining.ipynb', 'git_commit': 'not_available', 'status': summary['status'], 'primary_metric': 'selected_hard_negatives', 'primary_value': summary['selected_hard_negatives'], 'notes': 'Validation excluded. threshold={}; fallback_top_k={}.'.format(HARD_NEGATIVE_THRESHOLD, FALLBACK_TOP_K_PER_VIDEO)}
registry = registry.loc[~registry['run_id'].eq('V3_03_HARD_NEGATIVE_MINING_R1')]
registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
registry.to_csv(REGISTRY_PATH, index=False)

print(json.dumps(summary, ensure_ascii=False, indent=2))
if mining_complete and len(selected):
    display(selected[['sequence_id', 'video_id', 'window_start', 'window_end', 'positive_probability', 'selection_reason', 'sample_weight_hn1']].head(20))
    print('Contact sheet:', CONTACT_SHEET_PATH)
